# Ontology vs Semantic Model, Explained with My Running Data

Companion notebook to the article *Ontology vs Semantic Model, Explained with My Running Data* on [buckenhofer.com](https://buckenhofer.com). It uses DuckDB and Apache Ossie.

Almost everything here is SQL. Python only connects DuckDB, writes nothing but small glue code, and calls the LLM in the last part.

1. Data: my runs from 2021 to 2025 in DuckDB
2. Ontology, semantic model and mapping as Apache Ossie YAML, checked with the official validator
3. Queries through the semantic model
4. The same terms in Turtle/SKOS
5. Outlook: a knowledge graph with SQL/PGQ
6. Test: does the model help an LLM write correct SQL? Uses Colab's built-in Gemini, no API key needed.

A free CPU runtime is enough. Run the cells from top to bottom. To start over, delete `running.duckdb` or restart the runtime.

## Setup

In [37]:
%pip install -q duckdb==2.0.0 jupysql rdflib

In [38]:
import duckdb

con = duckdb.connect('running.duckdb')   # one database file, one connection
%load_ext sql
%sql con --alias duckdb
%config SqlMagic.displaylimit = 50
%config SqlMagic.feedback = 0

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [39]:
%%sql
INSTALL ossie FROM community;
LOAD ossie;
INSTALL duckpgq FROM community;
LOAD duckpgq;

Success


## 1. Data

My runs from Strava, 2021 to 2025: date, GPS distance and moving time, nothing else. Strava has no field for the type of run, so the data script assigns it: the five half marathons in Ulm by date, with their official net time, a long run from 18 km, everything else is an easy run.

In [40]:
#@title runs.csv (883 runs)
open('runs.csv', 'w').write('''run_id,run_date,distance_m,time_sec
1,2021-01-02,7028,2631
2,2021-01-04,10235,3774
3,2021-01-06,10827,3895
4,2021-01-08,8487,2833
5,2021-01-11,10794,3874
6,2021-01-14,9349,3570
7,2021-01-16,10050,3911
8,2021-01-18,10876,4181
9,2021-01-20,10086,3970
10,2021-01-21,8423,2841
11,2021-01-23,9188,2998
12,2021-01-25,9105,3506
13,2021-01-27,10409,3344
14,2021-01-28,10118,3658
15,2021-01-30,11386,4178
16,2021-02-01,11249,3999
17,2021-02-03,10130,3800
18,2021-02-04,8320,2848
19,2021-02-06,9961,3923
20,2021-02-08,10204,3938
21,2021-02-09,8093,2670
22,2021-02-10,8387,2976
23,2021-02-13,6613,2494
24,2021-02-15,8023,2802
25,2021-02-16,9735,3704
26,2021-02-17,8217,3147
27,2021-02-19,9089,3280
28,2021-02-20,10996,4336
29,2021-02-22,10418,3925
30,2021-02-23,8940,3192
31,2021-02-24,8202,3048
32,2021-02-26,9277,3477
33,2021-02-27,10713,4020
34,2021-03-01,10579,4086
35,2021-03-02,9963,3626
36,2021-03-03,10008,3520
37,2021-03-04,9675,3453
38,2021-03-06,9937,3862
39,2021-03-08,12957,4763
40,2021-03-10,10078,3683
41,2021-03-11,9401,3500
42,2021-03-12,8195,3042
43,2021-03-13,10099,3608
44,2021-03-15,13529,4882
45,2021-03-17,9991,3923
46,2021-03-18,10148,3815
47,2021-03-20,10545,3905
48,2021-03-22,14037,5034
49,2021-03-23,11016,3730
50,2021-03-25,13040,4768
51,2021-03-27,11088,4029
52,2021-03-29,14133,5102
53,2021-03-30,12035,4513
54,2021-04-01,12305,4613
55,2021-04-03,10909,4094
56,2021-04-05,12084,4409
57,2021-04-06,9819,3667
58,2021-04-08,12175,4371
59,2021-04-09,9915,3586
60,2021-04-10,9313,3132
61,2021-04-12,13342,4872
62,2021-04-14,12174,4527
63,2021-04-15,10221,3705
64,2021-04-17,12711,4443
65,2021-04-19,12555,4558
66,2021-04-20,10108,3680
67,2021-04-22,12256,4469
68,2021-04-24,11093,4163
69,2021-04-26,12751,4710
70,2021-04-27,10182,3673
71,2021-04-28,8930,3148
72,2021-04-30,11541,3908
73,2021-05-01,9507,3347
74,2021-05-03,13944,4858
75,2021-05-04,11772,4056
76,2021-05-06,11798,4129
77,2021-05-07,10274,3456
78,2021-05-08,11468,4089
79,2021-05-10,13628,5038
80,2021-05-11,10725,3944
81,2021-05-13,15846,5635
82,2021-05-15,13032,4493
83,2021-05-17,14109,5025
84,2021-05-18,10942,3826
85,2021-05-19,9759,3478
86,2021-05-21,12173,4293
87,2021-05-22,12835,4386
88,2021-05-24,15510,5507
89,2021-05-25,11953,4153
90,2021-05-28,14456,5125
91,2021-05-29,14037,4455
92,2021-05-31,11376,4018
93,2021-06-01,12371,4416
94,2021-06-03,17289,6009
95,2021-06-05,15130,5229
96,2021-06-07,13742,4818
97,2021-06-08,10243,3635
98,2021-06-11,11098,4029
99,2021-06-12,15076,5016
100,2021-06-14,12202,4537
101,2021-06-15,11386,4004
102,2021-06-17,12045,4488
103,2021-06-19,14782,5118
104,2021-06-21,13308,4402
105,2021-06-22,11518,3744
106,2021-06-24,11241,4034
107,2021-06-25,11466,3771
108,2021-06-26,11725,3984
109,2021-06-28,13295,4926
110,2021-06-30,12030,4219
111,2021-07-02,12337,4308
112,2021-07-03,14073,4896
113,2021-07-05,13364,4594
114,2021-07-06,12160,4261
115,2021-07-07,11546,4256
116,2021-07-10,14545,5057
117,2021-07-12,14031,5146
118,2021-07-13,10334,3799
119,2021-07-14,11312,3867
120,2021-07-14,2090,617
121,2021-07-16,10344,3667
122,2021-07-17,13163,4597
123,2021-07-19,13027,4789
124,2021-07-20,11939,4410
125,2021-07-21,9915,3579
126,2021-07-23,9841,3524
127,2021-07-24,11798,4066
128,2021-07-26,11986,4540
129,2021-07-27,11018,4021
130,2021-07-28,9526,3480
131,2021-07-30,8970,3171
132,2021-07-31,13289,4673
133,2021-08-02,11993,4303
134,2021-08-04,12172,4277
135,2021-08-06,10085,3547
136,2021-08-07,14386,4865
137,2021-08-09,11504,3844
138,2021-08-10,11166,3998
139,2021-08-11,13255,4375
140,2021-08-13,10827,3970
141,2021-08-14,13170,4740
142,2021-08-17,10675,3773
143,2021-08-19,10401,3766
144,2021-08-21,14108,5060
145,2021-08-23,13317,4717
146,2021-08-24,10218,3671
147,2021-08-25,10308,3772
148,2021-08-27,10347,3291
149,2021-08-28,16184,5700
150,2021-08-30,11144,3968
151,2021-09-01,10270,3334
152,2021-09-02,9169,3293
153,2021-09-04,15845,5284
154,2021-09-06,11206,4074
155,2021-09-08,9421,3089
156,2021-09-10,9916,3532
157,2021-09-11,13954,4866
158,2021-09-13,10922,4060
159,2021-09-14,8749,3224
160,2021-09-16,10254,3419
161,2021-09-18,16927,5643
162,2021-09-20,11128,4060
163,2021-09-22,9924,3214
164,2021-09-24,15343,5253
165,2021-09-25,13508,4640
166,2021-09-28,8398,2741
167,2021-09-30,7241,2580
168,2021-10-03,21442,6487
169,2021-10-09,14179,5156
170,2021-10-11,6131,2112
171,2021-10-14,7548,2611
172,2021-10-16,13928,4944
173,2021-10-18,10154,3568
174,2021-10-20,10282,3685
175,2021-10-22,9141,3269
176,2021-10-23,11978,4372
177,2021-10-25,10738,3862
178,2021-10-27,854,285
179,2021-10-27,7757,2738
180,2021-10-30,8841,3178
181,2021-11-01,13412,4704
182,2021-11-03,13033,4616
183,2021-11-06,9561,3414
184,2021-11-08,10358,3741
185,2021-11-10,10274,3713
186,2021-11-11,8346,2952
187,2021-11-13,8583,3223
188,2021-11-15,8325,3019
189,2021-11-18,10390,3690
190,2021-11-27,8612,3134
191,2021-11-30,9442,3506
192,2021-12-03,9009,3321
193,2021-12-06,1847,762
194,2021-12-06,2499,889
195,2021-12-06,4407,1544
196,2021-12-08,8206,3026
197,2021-12-11,6001,2221
198,2021-12-12,9421,3409
199,2021-12-14,8058,3036
200,2021-12-20,7949,2954
201,2021-12-22,7688,2753
202,2021-12-24,8026,2936
203,2021-12-27,9064,3297
204,2021-12-28,8989,3172
205,2021-12-30,8060,2831
206,2022-01-03,10174,3823
207,2022-01-05,8317,2896
208,2022-01-10,2101,844
209,2022-01-10,5223,1430
210,2022-01-12,8383,2931
211,2022-01-17,10217,3812
212,2022-01-19,9195,3365
213,2022-01-22,10870,3610
214,2022-01-24,10227,3852
215,2022-01-26,8778,3206
216,2022-01-28,11040,4093
217,2022-01-31,10678,3912
218,2022-02-02,8333,3067
219,2022-02-03,9162,3326
220,2022-02-05,7967,2936
221,2022-02-08,9394,3518
222,2022-02-10,9798,3628
223,2022-02-12,10551,4013
224,2022-02-25,8057,3085
225,2022-03-05,5754,2113
226,2022-03-10,8210,3113
227,2022-03-12,8040,2992
228,2022-03-14,9185,3447
229,2022-03-16,9734,3595
230,2022-03-19,9146,3609
231,2022-03-21,8204,3091
232,2022-03-23,7912,2934
233,2022-03-24,9241,3457
234,2022-03-26,8281,3091
235,2022-03-28,10449,3869
236,2022-03-30,11134,4217
237,2022-04-01,6763,2530
238,2022-04-06,8892,3389
239,2022-04-08,7653,2906
240,2022-04-11,7917,3019
241,2022-04-12,8419,3208
242,2022-04-14,9353,3230
243,2022-04-15,8646,3334
244,2022-04-16,6062,2231
245,2022-04-18,10399,3874
246,2022-04-19,8173,3008
247,2022-04-21,8183,3156
248,2022-04-23,6282,2339
249,2022-04-26,8244,3098
250,2022-04-30,7949,2948
251,2022-05-02,8606,3415
252,2022-05-03,8022,2730
253,2022-05-06,8054,2966
254,2022-05-07,8071,2949
255,2022-05-09,9117,3361
256,2022-05-11,10065,3686
257,2022-05-13,7076,2617
258,2022-05-14,8995,3337
259,2022-05-16,8176,3127
260,2022-05-17,8215,3034
261,2022-05-19,8552,3047
262,2022-05-21,7120,2494
263,2022-05-24,8165,3095
264,2022-05-26,10978,4137
265,2022-05-28,9138,3227
266,2022-05-30,10276,3853
267,2022-06-01,10324,3621
268,2022-06-02,7618,2771
269,2022-06-04,13137,4664
270,2022-06-06,13175,4487
271,2022-06-08,8959,3286
272,2022-06-09,8158,2635
273,2022-06-11,12268,4389
274,2022-06-13,8696,3040
275,2022-06-14,8794,3063
276,2022-06-16,10792,3747
277,2022-06-20,9409,3364
278,2022-06-21,7997,2838
279,2022-06-23,8175,2983
280,2022-06-25,9988,3580
281,2022-06-27,8696,3266
282,2022-06-29,6543,2284
283,2022-06-30,6738,2429
284,2022-07-02,9173,3301
285,2022-07-04,4177,1442
286,2022-07-08,7415,2683
287,2022-07-11,10094,3702
288,2022-07-13,10039,3680
289,2022-07-15,9190,3319
290,2022-07-16,10111,3744
291,2022-07-18,11043,3941
292,2022-07-19,8975,3304
293,2022-07-21,8489,3183
294,2022-07-25,10532,3900
295,2022-07-27,7259,2584
296,2022-07-28,7964,2903
297,2022-07-30,10519,3742
298,2022-08-01,10276,3707
299,2022-08-03,8877,2933
300,2022-08-04,8357,2987
301,2022-08-06,8949,3216
302,2022-08-08,7612,2901
303,2022-08-08,2544,1020
304,2022-08-09,8520,3056
305,2022-08-11,12147,4374
306,2022-08-13,10523,3406
307,2022-08-15,13067,4678
308,2022-08-17,9051,3262
309,2022-08-20,14151,4944
310,2022-08-22,10137,3848
311,2022-08-24,15822,5773
312,2022-08-25,8809,3113
313,2022-08-27,14183,5074
314,2022-08-29,11198,3735
315,2022-08-31,9871,3702
316,2022-09-01,8418,3088
317,2022-09-03,18070,6392
318,2022-09-05,10427,3901
319,2022-09-07,9940,3515
320,2022-09-10,18208,6505
321,2022-09-12,10286,3905
322,2022-09-15,7266,2480
323,2022-09-17,17223,6059
324,2022-09-18,7015,2151
325,2022-09-20,8846,3096
326,2022-09-22,5887,2080
327,2022-09-25,21821,6559
328,2022-09-28,7602,2543
329,2022-10-01,8857,3215
330,2022-10-03,12099,4304
331,2022-10-07,8868,3052
332,2022-10-08,13187,4639
333,2022-10-10,10261,3466
334,2022-10-12,7991,2925
335,2022-10-14,7612,2713
336,2022-10-17,10051,3709
337,2022-10-24,9416,3520
338,2022-10-25,8120,2961
339,2022-10-29,9157,3223
340,2022-10-31,10917,3891
341,2022-11-02,7297,2714
342,2022-11-07,9340,3398
343,2022-11-11,7211,2660
344,2022-11-12,8328,3088
345,2022-11-14,7880,3028
346,2022-11-15,9153,3316
347,2022-11-18,9030,2988
348,2022-11-22,8847,3381
349,2022-11-26,9222,3451
350,2022-11-28,9106,3425
351,2022-12-01,8731,3270
352,2022-12-05,9116,3358
353,2022-12-07,8125,2952
354,2022-12-10,5675,2013
355,2022-12-28,5700,2119
356,2022-12-30,5790,2184
357,2022-12-31,6158,2357
358,2023-01-02,9551,3622
359,2023-01-04,8056,3068
360,2023-01-06,7317,2653
361,2023-01-07,8692,3102
362,2023-01-09,7886,2933
363,2023-01-14,7343,2816
364,2023-01-16,9424,3506
365,2023-01-21,7554,2754
366,2023-01-23,9553,3483
367,2023-01-26,9159,3273
368,2023-01-28,9709,3469
369,2023-01-30,8965,3301
370,2023-02-01,8048,2866
371,2023-02-03,8952,3199
372,2023-02-04,9169,3172
373,2023-02-06,10221,3619
374,2023-02-08,6957,2473
375,2023-02-10,6110,2178
376,2023-02-11,9192,3174
377,2023-02-13,10116,3535
378,2023-02-15,10013,3453
379,2023-02-16,7031,2491
380,2023-02-18,8819,3152
381,2023-02-20,11062,3789
382,2023-02-22,9035,3195
383,2023-02-23,8681,3024
384,2023-02-25,8666,3140
385,2023-02-27,8985,3140
386,2023-03-01,8606,3084
387,2023-03-03,9134,3235
388,2023-03-04,8576,3053
389,2023-03-06,10203,3614
390,2023-03-08,9580,3346
391,2023-03-09,7911,2753
392,2023-03-11,8000,2911
393,2023-03-13,10896,3845
394,2023-03-14,8323,2961
395,2023-03-16,9715,3449
396,2023-03-18,8651,3089
397,2023-03-20,11562,3842
398,2023-03-21,8281,2861
399,2023-03-23,9617,3238
400,2023-03-25,9040,3213
401,2023-03-27,10621,3725
402,2023-03-29,10520,3768
403,2023-04-01,9201,3235
404,2023-04-03,10998,3706
405,2023-04-04,10345,3601
406,2023-04-06,10688,3647
407,2023-04-07,8170,2848
408,2023-04-08,8373,2898
409,2023-04-10,13332,4701
410,2023-04-11,7841,2787
411,2023-04-13,11551,3860
412,2023-04-14,7894,2757
413,2023-04-19,10129,3547
414,2023-04-21,9957,3426
415,2023-04-22,7642,2629
416,2023-04-24,10216,3518
417,2023-04-28,10032,3282
418,2023-04-29,8243,2958
419,2023-05-01,13480,4773
420,2023-05-03,10528,3641
421,2023-05-05,9151,3235
422,2023-05-06,9116,3201
423,2023-05-08,12202,4189
424,2023-05-09,8398,2957
425,2023-05-12,9922,3619
426,2023-05-13,9884,3404
427,2023-05-15,10425,3761
428,2023-05-17,8001,2901
429,2023-05-18,9134,3255
430,2023-05-20,11153,3987
431,2023-05-22,12293,4133
432,2023-05-23,10031,3505
433,2023-05-25,9095,3330
434,2023-05-27,14098,4829
435,2023-05-29,12195,4368
436,2023-05-30,9257,3264
437,2023-06-01,10953,3832
438,2023-06-03,11873,4303
439,2023-06-05,11353,3885
440,2023-06-06,10348,3560
441,2023-06-08,11293,3931
442,2023-06-10,8509,3041
443,2023-06-12,11210,4017
444,2023-06-14,9202,3318
445,2023-06-16,8570,3078
446,2023-06-17,9957,3529
447,2023-06-19,12503,4332
448,2023-06-20,8682,2784
449,2023-06-21,5870,1963
450,2023-06-22,7978,2702
451,2023-06-24,12027,4061
452,2023-06-26,12443,4158
453,2023-06-27,8987,3177
454,2023-06-29,10511,3619
455,2023-06-30,8990,3123
456,2023-07-03,12219,4087
457,2023-07-04,8267,2888
458,2023-07-07,9548,3482
459,2023-07-08,7345,2691
460,2023-07-10,13294,4505
461,2023-07-11,10139,3577
462,2023-07-13,9887,3472
463,2023-07-14,9173,3248
464,2023-07-17,10429,3625
465,2023-07-19,8061,2805
466,2023-07-20,13335,4547
467,2023-07-22,15190,5293
468,2023-07-25,9165,3362
469,2023-07-27,8777,3155
470,2023-07-29,7985,2875
471,2023-07-31,11057,4004
472,2023-08-02,10941,3516
473,2023-08-04,8146,2921
474,2023-08-05,14041,4830
475,2023-08-07,12292,4268
476,2023-08-08,9124,3019
477,2023-08-10,12182,3934
478,2023-08-12,16104,5775
479,2023-08-15,12272,4383
480,2023-08-17,18042,6257
481,2023-08-19,16360,5871
482,2023-08-21,2445,1255
483,2023-08-22,10115,3500
484,2023-08-24,10423,3739
485,2023-08-26,9742,3349
486,2023-08-28,12377,4210
487,2023-08-29,8210,2866
488,2023-08-31,19016,6859
489,2023-09-02,13078,4382
490,2023-09-04,10998,3713
491,2023-09-06,9068,3028
492,2023-09-09,17678,6089
493,2023-09-11,10165,3645
494,2023-09-13,11057,3518
495,2023-09-16,14173,5059
496,2023-09-18,11749,4063
497,2023-09-19,10144,3506
498,2023-09-21,10091,3393
499,2023-09-23,14977,5367
500,2023-09-25,10703,3600
501,2023-09-27,7621,2620
502,2023-10-01,21457,6327
503,2023-10-03,7103,2660
504,2023-10-05,9095,3201
505,2023-10-07,9280,3236
506,2023-10-09,9750,3332
507,2023-10-11,5649,3737
508,2023-10-12,8109,2850
509,2023-10-28,6834,2475
510,2023-10-30,9930,3528
511,2023-11-01,9000,3311
512,2023-11-04,10087,3604
513,2023-11-11,9742,3488
514,2023-11-14,8837,3177
515,2023-11-15,8456,2909
516,2023-11-18,9428,3363
517,2023-11-20,9797,3264
518,2023-11-21,7046,2447
519,2023-11-23,9346,3265
520,2023-11-25,7450,2731
521,2023-11-27,9061,3338
522,2023-11-30,8507,3058
523,2023-12-02,6964,2708
524,2023-12-05,10115,3611
525,2023-12-07,7515,2756
526,2023-12-09,9678,3242
527,2023-12-11,10593,3642
528,2023-12-14,9044,3306
529,2023-12-16,10172,3546
530,2023-12-20,11482,3905
531,2023-12-21,7505,2654
532,2023-12-23,10379,3466
533,2023-12-25,8186,2928
534,2023-12-27,9690,3460
535,2023-12-29,9030,3237
536,2024-01-01,9281,3403
537,2024-01-03,9991,3444
538,2024-01-05,10560,3467
539,2024-01-11,8970,3095
540,2024-01-13,7364,2352
541,2024-01-15,10391,3584
542,2024-01-18,10283,3523
543,2024-01-20,6245,2268
544,2024-01-22,10109,3602
545,2024-01-25,9538,3319
546,2024-01-25,14989,5615
547,2024-01-29,9660,3537
548,2024-01-31,8892,3028
549,2024-02-02,8164,2960
550,2024-02-03,8159,3106
551,2024-02-05,11441,4128
552,2024-02-06,8267,2942
553,2024-02-08,10933,3840
554,2024-02-10,10218,3386
555,2024-02-13,9554,3432
556,2024-02-15,8673,3876
557,2024-02-17,9474,3618
558,2024-02-22,8811,3220
559,2024-02-24,10448,3757
560,2024-02-27,8875,3217
561,2024-02-28,7487,2793
562,2024-02-29,14148,5084
563,2024-03-02,10415,3849
564,2024-03-05,9503,3506
565,2024-03-06,7123,2200
566,2024-03-08,8142,3159
567,2024-03-09,9213,3474
568,2024-03-11,10998,3811
569,2024-03-14,8814,3252
570,2024-03-16,10033,3748
571,2024-03-19,10144,3753
572,2024-03-20,7872,2876
573,2024-03-22,11086,3606
574,2024-03-23,10040,3794
575,2024-03-25,9173,3477
576,2024-03-26,10200,3613
577,2024-03-28,13800,4824
578,2024-03-30,9356,3406
579,2024-04-01,12127,4405
580,2024-04-03,8833,3153
581,2024-04-05,8155,2919
582,2024-04-06,9856,3516
583,2024-04-08,12151,4344
584,2024-04-09,6930,2722
585,2024-04-13,11311,3967
586,2024-04-15,10615,3674
587,2024-04-19,14314,4870
588,2024-04-23,9891,3409
589,2024-04-24,7970,2723
590,2024-04-26,11251,3877
591,2024-04-27,7819,2705
592,2024-04-29,12267,3856
593,2024-04-30,9041,3049
594,2024-05-04,10656,3626
595,2024-05-06,11426,3892
596,2024-05-09,10619,3716
597,2024-05-11,10206,3624
598,2024-05-13,11995,4182
599,2024-05-15,10170,3467
600,2024-05-18,9231,3310
601,2024-05-20,15101,5362
602,2024-05-22,8263,2905
603,2024-05-23,8185,2801
604,2024-05-25,10101,2902
605,2024-05-27,12154,4304
606,2024-05-30,10375,3738
607,2024-06-01,10519,3517
608,2024-06-03,11393,3992
609,2024-06-06,8075,2858
610,2024-06-08,8585,3012
611,2024-06-12,11798,4180
612,2024-06-13,9966,3701
613,2024-06-14,3676,1506
614,2024-06-15,10210,3578
615,2024-06-18,10377,3833
616,2024-06-20,9136,3121
617,2024-06-22,11095,3714
618,2024-06-24,13125,4531
619,2024-06-26,9311,3298
620,2024-06-28,8532,3362
621,2024-06-29,8066,2933
622,2024-07-01,13617,4955
623,2024-07-02,4591,1804
624,2024-07-04,9583,3157
625,2024-07-06,9521,3148
626,2024-07-08,13989,5164
627,2024-07-10,8959,3282
628,2024-07-12,5992,2283
629,2024-07-13,9081,3144
630,2024-07-15,15126,5603
631,2024-07-17,9908,3296
632,2024-07-19,6404,2389
633,2024-07-20,9177,3118
634,2024-07-22,10970,3837
635,2024-07-24,7408,2733
636,2024-07-26,8359,2997
637,2024-07-27,8654,3051
638,2024-07-29,12591,4627
639,2024-07-30,1539,667
640,2024-07-31,8680,2917
641,2024-08-02,10252,3381
642,2024-08-03,8179,3017
643,2024-08-05,15287,5566
644,2024-08-07,8210,3024
645,2024-08-08,8187,2975
646,2024-08-10,9849,3311
647,2024-08-12,18376,6614
648,2024-08-14,9496,3216
649,2024-08-17,10128,3342
650,2024-08-19,10547,3766
651,2024-08-21,8822,2965
652,2024-08-23,8481,2974
653,2024-08-24,7022,2733
654,2024-08-26,18990,6874
655,2024-08-28,10139,3395
656,2024-08-30,10126,3427
657,2024-08-31,7221,2627
658,2024-09-02,20094,7460
659,2024-09-05,10301,3491
660,2024-09-07,10080,3615
661,2024-09-09,21595,7961
662,2024-09-11,10139,3339
663,2024-09-12,7002,2494
664,2024-09-14,10272,3514
665,2024-09-16,13913,4945
666,2024-09-17,5248,2101
667,2024-09-18,9005,3015
668,2024-09-19,5396,2027
669,2024-09-21,8509,2992
670,2024-09-23,8413,2920
671,2024-09-29,21585,6466
672,2024-10-03,6215,2297
673,2024-10-08,8886,3080
674,2024-10-10,8361,2936
675,2024-10-11,7029,2520
676,2024-10-12,6466,2232
677,2024-10-14,9451,3396
678,2024-10-15,7509,2542
679,2024-10-17,9117,3264
680,2024-10-19,8133,2931
681,2024-10-21,9845,3492
682,2024-10-23,9474,3383
683,2024-10-24,6172,2184
684,2024-10-26,9929,3540
685,2024-10-28,10216,3429
686,2024-10-31,11127,3604
687,2024-11-02,6881,2434
688,2024-11-05,7482,2658
689,2024-11-07,6993,2445
690,2024-11-08,6163,2158
691,2024-11-09,7886,2769
692,2024-11-11,10138,3654
693,2024-11-13,9207,3292
694,2024-11-16,7526,2409
695,2024-11-18,7785,2615
696,2024-11-19,7528,2533
697,2024-11-21,8325,2800
698,2024-11-23,8246,2919
699,2024-11-25,10141,3612
700,2024-11-27,8084,2919
701,2024-11-28,8086,2964
702,2024-11-30,9059,3244
703,2024-12-02,10912,3869
704,2024-12-03,8989,3148
705,2024-12-05,9371,3255
706,2024-12-07,11268,3950
707,2024-12-09,5145,1850
708,2024-12-10,8661,2924
709,2024-12-12,11776,4097
710,2024-12-14,8971,3296
711,2024-12-16,9379,3467
712,2024-12-17,7981,2886
713,2024-12-19,8530,2939
714,2024-12-20,8767,3153
715,2024-12-21,7092,2626
716,2024-12-23,6409,2359
717,2024-12-24,7987,2835
718,2024-12-26,8044,2680
719,2024-12-28,8242,2820
720,2024-12-30,9063,3213
721,2024-12-31,6912,2395
722,2025-01-01,6064,2067
723,2025-01-03,9445,3105
724,2025-01-06,10861,3872
725,2025-01-11,10180,3578
726,2025-01-17,9299,3217
727,2025-01-18,7436,2659
728,2025-01-20,9145,3214
729,2025-01-24,8440,2984
730,2025-01-25,10313,3546
731,2025-01-27,11052,3907
732,2025-01-31,9222,3130
733,2025-02-01,8084,2646
734,2025-02-03,9676,3435
735,2025-02-05,8259,2854
736,2025-02-07,9040,3183
737,2025-02-08,8270,2947
738,2025-02-10,10093,3512
739,2025-02-13,10070,3573
740,2025-02-15,9155,3319
741,2025-02-17,9629,3458
742,2025-02-18,8465,2902
743,2025-02-24,10125,3817
744,2025-02-27,9013,3238
745,2025-02-28,7606,2770
746,2025-03-01,6231,2295
747,2025-03-03,7466,2799
748,2025-03-05,8653,3008
749,2025-03-07,5741,2121
750,2025-03-08,6915,2612
751,2025-03-10,10871,4104
752,2025-03-12,9096,3105
753,2025-03-14,8788,3032
754,2025-03-15,5121,1863
755,2025-03-17,8035,2824
756,2025-03-19,5437,1958
757,2025-03-21,7008,2413
758,2025-03-22,5456,2038
759,2025-03-24,11247,4315
760,2025-03-26,9069,3055
761,2025-03-28,8928,2934
762,2025-03-29,5984,2193
763,2025-03-31,13335,4890
764,2025-04-02,8846,2982
765,2025-04-04,8185,2947
766,2025-04-07,12307,4522
767,2025-04-09,9413,3096
768,2025-04-11,8018,2685
769,2025-04-12,7192,2657
770,2025-04-14,10573,3603
771,2025-04-16,7349,2646
772,2025-04-21,12139,4642
773,2025-04-25,9764,3216
774,2025-04-28,11405,4361
775,2025-04-30,8850,3022
776,2025-05-02,7536,2642
777,2025-05-03,5725,2076
778,2025-05-05,11549,4027
779,2025-05-09,7901,2770
780,2025-05-10,5239,1951
781,2025-05-12,10268,3680
782,2025-05-13,5148,1920
783,2025-05-16,9377,3084
784,2025-05-17,5947,2212
785,2025-05-19,10249,3695
786,2025-05-20,5945,2095
787,2025-05-23,7432,2791
788,2025-05-24,7821,2876
789,2025-05-26,9013,3243
790,2025-05-28,7481,2727
791,2025-05-30,9476,3036
792,2025-05-31,7052,2529
793,2025-06-06,8861,3142
794,2025-06-07,5816,2088
795,2025-06-09,11686,4211
796,2025-06-11,8029,2616
797,2025-06-13,8287,2955
798,2025-06-16,12084,4292
799,2025-06-18,8100,2943
800,2025-06-19,8479,3012
801,2025-06-21,9569,3321
802,2025-06-23,12909,4250
803,2025-06-25,8540,3056
804,2025-06-26,6069,2100
805,2025-06-28,8081,2912
806,2025-06-30,12104,4270
807,2025-07-02,8857,2829
808,2025-07-04,9246,3007
809,2025-07-05,5197,1895
810,2025-07-07,14652,5127
811,2025-07-09,5657,2048
812,2025-07-11,8205,2728
813,2025-07-12,5972,2091
814,2025-07-14,2052,1007
815,2025-07-15,9403,3405
816,2025-07-16,5155,1862
817,2025-07-18,8780,2828
818,2025-07-19,5137,1866
819,2025-07-21,6115,2088
820,2025-07-23,6772,2129
821,2025-07-26,7576,2718
822,2025-07-28,9531,3426
823,2025-08-01,7364,2270
824,2025-08-03,7879,2630
825,2025-08-05,4640,1359
826,2025-08-09,9187,3342
827,2025-08-11,14828,5548
828,2025-08-13,9209,3216
829,2025-08-15,9424,3302
830,2025-08-16,4090,1518
831,2025-08-18,8747,3021
832,2025-08-22,6679,2455
833,2025-08-23,6172,2350
834,2025-08-25,13771,5341
835,2025-08-28,9286,3508
836,2025-08-30,9884,3725
837,2025-09-01,16783,6292
838,2025-09-03,8922,2997
839,2025-09-05,7898,2727
840,2025-09-08,17204,6788
841,2025-09-10,9579,3241
842,2025-09-11,5893,2219
843,2025-09-13,17955,6820
844,2025-09-15,7355,2366
845,2025-09-17,5747,1866
846,2025-09-19,11034,3689
847,2025-09-20,6872,2581
848,2025-09-22,9622,3231
849,2025-09-24,8389,2979
850,2025-09-28,21267,6663
851,2025-10-02,7563,2763
852,2025-10-04,7689,2842
853,2025-10-06,8905,3300
854,2025-10-08,9175,3162
855,2025-10-11,10238,3762
856,2025-10-13,10300,3777
857,2025-10-15,7929,2749
858,2025-10-17,9080,3228
859,2025-10-20,11158,4020
860,2025-10-21,7474,2684
861,2025-10-23,11283,4116
862,2025-10-25,6674,2479
863,2025-11-05,7107,2431
864,2025-11-08,5846,2146
865,2025-11-11,8271,2986
866,2025-11-13,6671,2448
867,2025-11-17,3853,1426
868,2025-11-29,5788,2073
869,2025-12-03,5639,1939
870,2025-12-05,6073,2239
871,2025-12-06,7101,2610
872,2025-12-08,8121,3077
873,2025-12-11,8208,2888
874,2025-12-13,7244,2665
875,2025-12-15,7169,2682
876,2025-12-17,9881,3411
877,2025-12-19,6074,2147
878,2025-12-20,5344,2000
879,2025-12-23,8291,2913
880,2025-12-24,5002,1910
881,2025-12-27,8064,2978
882,2025-12-29,7171,2680
883,2025-12-31,5369,1899
''')

22353

In [41]:
%%sql
-- My runs 2021 to 2025 from Strava: date, GPS distance and moving time per run (runs.csv)

CREATE OR REPLACE TABLE run_types (
    run_type_id INTEGER PRIMARY KEY,
    run_type    VARCHAR NOT NULL,
    category    VARCHAR NOT NULL        -- Training or Race
);
INSERT INTO run_types VALUES
    (1, 'Easy run',      'Training'),
    (2, 'Long run',      'Training'),
    (3, 'Half marathon', 'Race');

CREATE OR REPLACE TABLE runs (
    run_id      INTEGER PRIMARY KEY,
    run_date    DATE    NOT NULL,
    run_type_id INTEGER NOT NULL REFERENCES run_types,
    distance_m  INTEGER NOT NULL,       -- measured by GPS
    time_sec    INTEGER NOT NULL        -- moving time, official net time for races
);

-- Strava has no field for the type of run. Races are known by date, with their official net time.
INSERT INTO runs
WITH half_marathons (run_date, official_sec) AS (
    VALUES (DATE '2021-10-03', 6576), (DATE '2022-09-25', 6554), (DATE '2023-10-01', 6322),
           (DATE '2024-09-29', 6458), (DATE '2025-09-28', 6700)
)
SELECT s.run_id, s.run_date,
       CASE WHEN h.run_date IS NOT NULL THEN 3
            WHEN s.distance_m >= 18000  THEN 2
            ELSE 1 END,
       s.distance_m,
       coalesce(h.official_sec, s.time_sec)
FROM read_csv('runs.csv') s
LEFT JOIN half_marathons h ON h.run_date = s.run_date AND s.distance_m > 20000   -- the race, not a warm-up
ORDER BY s.run_id;

RuntimeError: Catalog Error: Could not drop the table because this table is main key table of the table "runs"


In [ ]:
%%sql
SELECT r.run_type, count(*) AS runs, round(sum(distance_m) / 1000) AS km
FROM runs JOIN run_types r USING (run_type_id)
GROUP BY ALL ORDER BY runs DESC;

## 2. Ontology, semantic model and mapping

The ontology describes *what* the terms mean, independent of tables. The semantic model describes *how* to calculate metrics from the tables. The mapping connects both.

In [ ]:
%%writefile running.ontology.yaml
version: 0.2.0.dev0
name: Running
description: Terms around my runs, independent of any table
prefixes:
  run: https://buckenhofer.com/ontology/running#
ontology:
  - concept: Kilometers
    type: ValueType
    extends: [Decimal]
    requires: ["Kilometers > 0"]

  - concept: Minutes
    type: ValueType
    extends: [Decimal]
    requires: ["Minutes > 0"]

  - concept: Run
    type: EntityType
    description: A recorded activity on foot
    iri: run:Run
    identify_by: [nr]
    relationships:
      - name: nr
        roles: [concept: Integer]
        multiplicity: OneToOne
        verbalizes: ["{Run} has number {Integer}"]
      - name: date
        roles: [concept: Date]
        multiplicity: ManyToOne
        verbalizes: ["{Run} took place on {Date}"]
      - name: distance
        roles: [concept: Kilometers]
        multiplicity: ManyToOne
        verbalizes: ["{Run} is {Kilometers} long"]
      - name: duration
        roles: [concept: Minutes]
        multiplicity: ManyToOne
        verbalizes: ["{Run} takes {Minutes}"]
      - name: type
        roles: [concept: String]
        multiplicity: ManyToOne
        verbalizes: ["{Run} is of type {String}"]

  - concept: Training
    type: EntityType
    extends: [Run]
    iri: run:Training

  - concept: Race
    type: EntityType
    extends: [Run]
    description: Official race with bib number and timing
    iri: run:Race

  - concept: EasyRun
    type: EntityType
    extends: [Training]
    derived_by: ["Run.type == 'Easy run'"]
    iri: run:EasyRun

  - concept: LongRun
    type: EntityType
    extends: [Training]
    description: Long, easy run
    derived_by: ["Run.type == 'Long run'"]
    requires: ["Run.distance >= 18"]
    iri: run:LongRun

  - concept: HalfMarathon
    type: EntityType
    extends: [Race]
    description: Race over the official distance of 21.0975 km. The GPS watch usually measures a bit more.
    derived_by: ["Run.type == 'Half marathon'"]
    iri: run:HalfMarathon

In [ ]:
%%writefile running.semantic_model.yaml
version: 0.2.0.dev0
name: running
description: My runs from Strava, one run per row
ai_context:
  instructions: >
    Always filter half marathons and other races via run_types,
    never by distance. A long training run can be longer than 21 km.

datasets:
  - name: runs
    source: running.main.runs
    primary_key: [run_id]
    fields:
      - name: run_id
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: run_id}]}
      - name: run_date
        datatype: Date
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: run_date}]}
      - name: year
        datatype: Integer
        dimension: {is_time: true}
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: YEAR(run_date)}]}
      - name: run_type_id
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: run_type_id}]}
      - name: distance_m
        description: Distance measured by GPS in meters
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: distance_m}]}
      - name: time_sec
        description: Moving time in seconds, official net time for races
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: time_sec}]}

  - name: run_types
    source: running.main.run_types
    primary_key: [run_type_id]
    fields:
      - name: run_type_id
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: run_type_id}]}
      - name: run_type
        description: Easy run, Long run or Half marathon
        ai_context: {synonyms: [type of run, HM, half]}
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: run_type}]}
      - name: category
        description: Training or Race
        expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: category}]}

relationships:
  - name: run_has_type
    from: runs
    to: run_types
    from_columns: [run_type_id]
    to_columns: [run_type_id]

metrics:
  - name: run_count
    expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: COUNT(runs.run_id)}]}

  - name: distance_km
    expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: SUM(runs.distance_m) / 1000.0}]}

  - name: pace_sec_per_km
    description: Seconds per kilometer, total time divided by total distance, not the average of individual paces
    ai_context: {synonyms: [pace, average pace]}
    expression:
      dialects:
        - dialect: OSSIE_SQL_2026
          expression: SUM(runs.time_sec) / (SUM(runs.distance_m) / 1000.0)

  - name: best_time_sec
    expression: {dialects: [{dialect: OSSIE_SQL_2026, expression: MIN(runs.time_sec)}]}

In [ ]:
%%writefile running.mapping.yaml
version: 0.2.0.dev0
name: running_mapping
description: Connects the terms of the ontology with the fields of the semantic model
ontology_ref:
  name: Running
  iri: ./running.ontology.yaml
semantic_model_ref:
  name: running
  iri: ./running.semantic_model.yaml
concept_mappings:
  - concept: Run
    object_mappings:
      - expression: runs.run_id
    link_mappings:
      - object_mapping: {expression: runs.run_id}
        children:
          - relationship: date
            object_mapping: {concept: Date, expression: runs.run_date}
          - relationship: distance
            object_mapping: {concept: Kilometers, expression: runs.distance_m / 1000.0}
          - relationship: duration
            object_mapping: {concept: Minutes, expression: runs.time_sec / 60.0}
          - relationship: type
            object_mapping: {concept: String, expression: run_types.run_type}

Check all three files with the validator from the Apache Ossie repository:

In [ ]:
!git clone -q --depth 1 https://github.com/apache/ossie.git 2>/dev/null || true
!pip install -q -U jsonschema pyyaml "sqlglot>=30" referencing   # the validator needs sqlglot 30 or newer
!python ossie/validation/validate.py running.semantic_model.yaml
!python ossie/validation/validate.py --schema ossie/ontology/ontology.json running.ontology.yaml
!python ossie/validation/validate.py --schema ossie/ontology/mapping.json running.mapping.yaml

## 3. Queries through the semantic model

`ossie_query(metrics, dimensions, filters)` takes three lists. The model adds joins and `GROUP BY`.

In [ ]:
%%sql
CALL ossie_load('running.semantic_model.yaml');

In [ ]:
%%sql
SELECT name, description FROM ossie_metrics();

**Question 1: How fast did I run on average?** Right through the model, wrong with a naive average of paces:

In [ ]:
%%sql
SELECT to_seconds(round(pace_sec_per_km)) AS pace
FROM ossie_query(['pace_sec_per_km']);

In [ ]:
%%sql
SELECT to_seconds(round(avg(time_sec / (distance_m / 1000.0)))) AS pace_wrong
FROM runs;

**Question 2: How many half marathons?** Counting by distance also catches a long training run:

In [ ]:
%%sql
SELECT count(*) AS half_marathons_naive FROM runs WHERE distance_m >= 21097.5;

In [ ]:
%%sql
SELECT run_date, distance_m FROM runs WHERE distance_m >= 21097.5 ORDER BY run_date;

In [ ]:
%%sql
SELECT * FROM ossie_query(['run_count'], [], ['run_types.run_type = ''Half marathon''']);

**Question 3: My half marathons in Ulm**

In [ ]:
%%sql
SELECT "runs.year"                          AS year,
       to_seconds(best_time_sec)            AS finish_time,
       to_seconds(round(pace_sec_per_km))   AS pace
FROM ossie_query(['best_time_sec', 'pace_sec_per_km'],
                 ['runs.year'],
                 ['run_types.run_type = ''Half marathon''']);

**The SQL behind a query**, without running it:

In [ ]:
%%sql
SELECT ossie_compile(['pace_sec_per_km'], ['run_types.category'], []) AS generated_sql;

## 4. The same terms in Turtle/SKOS

SKOS is the W3C standard for simple concept schemes. The IRIs are the same as in the `iri` fields of the Ossie ontology. SPARQL needs a small Python library, so this is the one cell with a bit more Python.

In [ ]:
%%writefile running.ttl
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix run:  <https://buckenhofer.com/ontology/running#> .

run:Running a skos:ConceptScheme ;
    skos:prefLabel "Running"@en ;
    skos:hasTopConcept run:Run .

run:Run a skos:Concept ;
    skos:inScheme run:Running ;
    skos:prefLabel "Run"@en ;
    skos:definition "A recorded activity on foot"@en .

run:Training a skos:Concept ;
    skos:inScheme run:Running ;
    skos:prefLabel "Training"@en ;
    skos:broader run:Run .

run:Race a skos:Concept ;
    skos:inScheme run:Running ;
    skos:prefLabel "Race"@en ;
    skos:definition "Official race with bib number and timing"@en ;
    skos:broader run:Run .

run:EasyRun a skos:Concept ;
    skos:inScheme run:Running ;
    skos:prefLabel "Easy run"@en ;
    skos:broader run:Training .

run:LongRun a skos:Concept ;
    skos:inScheme run:Running ;
    skos:prefLabel "Long run"@en ;
    skos:broader run:Training .

run:HalfMarathon a skos:Concept ;
    skos:inScheme run:Running ;
    skos:prefLabel "Half marathon"@en ;
    skos:altLabel "HM"@en, "half"@en ;
    skos:definition "Race over the official distance of 21.0975 km"@en ;
    skos:broader run:Race .

In [ ]:
from rdflib import Graph

skos = Graph().parse("running.ttl")
for row in skos.query("""
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT ?broader WHERE {
      ?concept skos:altLabel "HM"@en ;
               skos:broader+ ?b .
      ?b skos:prefLabel ?broader .
    }"""):
    print(row.broader)

## 5. Outlook: a knowledge graph with SQL/PGQ

Ontology plus data gives a knowledge graph. The concepts and "is a" edges come from the ontology, the edge from each run to its concept comes from the mapping. DuckPGQ puts a graph view on top of the tables.

In [ ]:
%%sql
-- Outlook knowledge graph: ontology and data as one graph in DuckDB
-- Concepts and "is a" edges come from running.ontology.yaml (extends)
CREATE OR REPLACE TABLE concepts (concept VARCHAR PRIMARY KEY);
INSERT INTO concepts VALUES
    ('Run'), ('Training'), ('Race'),
    ('Easy run'), ('Long run'), ('Half marathon');

CREATE OR REPLACE TABLE is_a (sub_concept VARCHAR, super_concept VARCHAR);
INSERT INTO is_a VALUES
    ('Training', 'Run'), ('Race', 'Run'),
    ('Easy run', 'Training'), ('Long run', 'Training'),
    ('Half marathon', 'Race');

-- Edges from each run to its concept, as described by the mapping
CREATE OR REPLACE TABLE run_is AS
SELECT r.run_id, t.run_type AS concept
FROM runs r JOIN run_types t USING (run_type_id);

DuckPGQ's syntax has to be switched on per connection. `%%sql` opens a new cursor for every statement, so the graph cells call `con.sql()` on the main connection instead. It is still plain SQL.

In [ ]:
con.execute("SET active_grammar_extensions = ['duckpgq']")
con.execute("""
-- Graph view on the tables with SQL/PGQ (DuckPGQ)

CREATE OR REPLACE PROPERTY GRAPH running_graph
VERTEX TABLES (runs, concepts)
EDGE TABLES (
    run_is SOURCE KEY (run_id) REFERENCES runs (run_id)
           DESTINATION KEY (concept) REFERENCES concepts (concept),
    is_a   SOURCE KEY (sub_concept) REFERENCES concepts (concept)
           DESTINATION KEY (super_concept) REFERENCES concepts (concept)
);
""")

In [ ]:
con.sql("""
-- Which runs are races?
FROM GRAPH_TABLE (running_graph
    MATCH (r:runs)-[e:run_is]->(c:concepts)
          -[i:is_a]->{1,3}(s:concepts WHERE s.concept = 'Race')
    COLUMNS (r.run_date, c.concept)
)
ORDER BY run_date;
""").df()

In [ ]:
con.sql("""
-- How many runs belong to each broader concept?
SELECT super_concept, count(*) AS runs
FROM GRAPH_TABLE (running_graph
    MATCH (r:runs)-[e:run_is]->(c:concepts)-[i:is_a]->{1,3}(s:concepts)
    COLUMNS (s.concept AS super_concept)
)
GROUP BY super_concept
ORDER BY runs DESC;
""").df()

## 6. Test: does the model help an LLM?

Eight questions with known answers, each under four conditions:

- **A, schema only, free SQL:** the LLM sees the table schema and writes any SQL.
- **B, semantic model, metric API:** the LLM sees the semantic model without its `ai_context` blocks and may read data only through `ossie_query`. Descriptions, metrics and relationships stay in.
- **C, semantic model with AI instructions, metric API:** like B, plus the `ai_context` blocks with synonyms and instructions. They repeat selected domain knowledge by hand; the ontology itself is not part of this context.
- **D, schema plus ontology and mapping, free SQL:** the LLM sees schema, ontology and mapping and writes any SQL.

Each answer runs in DuckDB, wrapped as a subquery. That rules out DDL and multiple statements in practice, but it is not a security sandbox. After an error, the LLM gets the error message and one more try. Every attempt is stored. A refusal by Ossie counts as *refused*, not as wrong.

Each question has its own check. A count or a sum must come back as exactly one row with the expected value (0.5 % tolerance, 0.2 % for the pace, because right and wrong are only 1.3 seconds apart). The finish times must come back as five rows, each with the right year next to its time (3 seconds tolerance, because the times are only 22 seconds apart in places).

The LLM is Colab's built-in Gemini by default. Colab limits how much it can be used; when calls fail with error 403, switch `LLM` to `'gemini_api'` with your own key from Google AI Studio, or to `'openrouter'` for any model on OpenRouter, then rerun with `RESUME = True` to keep what is already done. For another provider, write your own `ask_llm`. Results are stored per model, so several runs with different models can be compared in the same tables.

In [42]:
LLM = 'openrouter'                       # 'colab' = Colab's built-in Gemini, no key
                                    # 'gemini_api' = Gemini API, secret GEMINI_API_KEY
                                    # 'openrouter' = any model on openrouter.ai, secret OPENROUTER_API_KEY
MODEL = 'z-ai/glm-5.1' #'meta-llama/llama-3.1-8b-instruct'   # for 'colab' also None = Colab default; for 'openrouter' an id from
                                    # openrouter.ai/models, e.g. 'meta-llama/llama-3.1-8b-instruct'
RUNS = 5          # repetitions per question and condition, as in the article; 1 is a quick smoke test
PARALLEL = 4      # concurrent LLM calls; lower it if the log shows many retries
RESUME = False    # True: keep the finished tasks of this model and run only the missing ones

MODEL_LABEL = MODEL or "colab-default"

if LLM == 'colab':
    from google.colab import ai
    print(ai.list_models())         # models available to your account

    def ask_llm(prompt: str) -> str:
        return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

elif LLM == 'gemini_api':
    # Key from aistudio.google.com, stored as Colab secret GEMINI_API_KEY (key icon on the left, notebook access on)
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=True)
    from google import genai
    from google.colab import userdata
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"),
                          http_options=genai.types.HttpOptions(timeout=180_000))   # ms; a hanging call fails and is retried
    API_MODEL = MODEL.removeprefix("google/")

    def ask_llm(prompt: str) -> str:
        return client.models.generate_content(model=API_MODEL, contents=prompt).text

elif LLM == 'openrouter':
    # OpenAI-compatible API; key from openrouter.ai/keys, stored as Colab secret OPENROUTER_API_KEY
    import requests
    from google.colab import userdata
    OPENROUTER_KEY = userdata.get("OPENROUTER_API_KEY")

    def ask_llm(prompt: str) -> str:
        r = requests.post("https://openrouter.ai/api/v1/chat/completions", timeout=180,
                          headers={"Authorization": f"Bearer {OPENROUTER_KEY}"},
                          json={"model": MODEL, "messages": [{"role": "user", "content": prompt}]})
        r.raise_for_status()   # rate limits and server errors are retried by the loop
        return r.json()["choices"][0]["message"]["content"] or ""

from time import monotonic
started = monotonic()
print("Testing with:", MODEL_LABEL, "via", LLM, "| test call:", ask_llm("Reply with OK only.").strip()[:20],
      f"in {monotonic() - started:.1f} s")   # fails here if key, model name or quota is wrong

Testing with: z-ai/glm-5.1 via openrouter | test call: OK in 8.1 s


Questions, check type, expected values, tolerance and the plausible wrong answer (the trap). A *value* check accepts equivalent values, for example pace in seconds or in minutes per kilometer. A *year_pairs* check lists year, time in seconds and time in minutes.

In [43]:
%%sql
CREATE SCHEMA IF NOT EXISTS eval;

CREATE OR REPLACE TABLE eval.questions AS
SELECT * FROM (VALUES
    ('pace',       'What was my average pace across all runs?',         'value', [[355.547, 5.9258]], 0.002, 'average of individual paces: 356.8 s/km'),
    ('pace_speed', 'How fast was I on average per kilometer?',          'value', [[355.547, 5.9258]], 0.002, 'average of individual paces: 356.8 s/km'),
    ('hm',         'How many half marathons have I run?',               'value', [[5]],               0.005, 'counting runs of 21.0975 km or more: 6'),
    ('hm_short',   'How often did I run an HM?',                        'value', [[5]],               0.005, 'filtering on the value HM: 0'),
    ('races',      'How many races did I take part in?',                'value', [[5]],               0.005, 'no matching column on the run itself'),
    ('hm_times',   'What were my half marathon finish times, by year?', 'year_pairs',
                   [[2021, 6576, 109.6], [2022, 6554, 109.2333], [2023, 6322, 105.3667], [2024, 6458, 107.6333], [2025, 6700, 111.6667]],
                   0.0005, 'runs of 21.0975 km or more include a training run in 2024'),
    ('long_runs',  'How many long runs did I do?',                      'value', [[8]],               0.005, 'counting runs of 18 km or more includes the races: 13'),
    ('km_2024',    'How many kilometers did I run in 2024?',            'value', [[1787.686]],        0.005, 'control question without a trap')
) t(question_id, question, check_type, expected, tol, trap);

Count
8


The context per condition is built in SQL from the database catalog and the YAML files. Condition B removes every `ai_context` block with two regular expressions.

In [44]:
%%sql
CREATE OR REPLACE TABLE eval.contexts AS
WITH src AS (
    SELECT (SELECT string_agg(sql, chr(10) ORDER BY table_name) FROM duckdb_tables()
            WHERE database_name = 'running' AND table_name IN ('runs', 'run_types')) AS ddl,
           (SELECT content FROM read_text('running.semantic_model.yaml')) AS model,
           (SELECT content FROM read_text('running.ontology.yaml'))       AS ontology,
           (SELECT content FROM read_text('running.mapping.yaml'))        AS mapping
),
parts AS (
    SELECT 'The database schema:' || chr(10) || '```sql' || chr(10) || ddl || chr(10) || '```' AS schema_text,
           regexp_replace(regexp_replace(model, '(?m)^ai_context:\n(?:  .*\n)+', '', 'g'),
                          '(?m)^ *ai_context: \{[^\n]*\}\n', '', 'g') AS model_without_ai,
           model, ontology, mapping,
           'Answer only with exactly one DuckDB SQL query in a ```sql block.' AS sql_format,
           'Answer only with exactly one DuckDB SQL query in a ```sql block. '
           || 'Read data only through the table function ossie_query(metrics, dimensions, filters); '
           || 'direct access to tables is not allowed. metrics is a list of metric names, '
           || 'dimensions a list of dataset.field, filters a list of SQL conditions on dataset.field, '
           || 'for example ossie_query([''metric_a''], [''dataset.field''], [''dataset.field = 1'']). '
           || 'Result columns are named like the metrics and dimensions, e.g. "dataset.field". '
           || 'Several ossie_query calls in one query are allowed.' AS ossie_format
    FROM src
)
SELECT 'A' AS condition, schema_text || chr(10) || chr(10) || sql_format AS context FROM parts
UNION ALL
SELECT 'B', 'The semantic model (Apache Ossie):' || chr(10) || '```yaml' || chr(10) || model_without_ai || '```'
            || chr(10) || chr(10) || ossie_format FROM parts
UNION ALL
SELECT 'C', 'The semantic model (Apache Ossie):' || chr(10) || '```yaml' || chr(10) || model || '```'
            || chr(10) || chr(10) || ossie_format FROM parts
UNION ALL
SELECT 'D', schema_text || chr(10) || chr(10) || 'The ontology and the mapping (Apache Ossie):' || chr(10)
            || '```yaml' || chr(10) || ontology || '```' || chr(10) || '```yaml' || chr(10) || mapping || '```'
            || chr(10) || chr(10) || sql_format FROM parts;

Count
4


In [45]:
%%sql
CREATE OR REPLACE TABLE eval.tasks AS
SELECT row_number() OVER (ORDER BY run, c.condition, q.question_id) AS task_id,
       c.condition, q.question_id, run,
       'You answer questions about my running data in a DuckDB database. Follow the answer format exactly.'
       || chr(10) || chr(10) || c.context || chr(10) || chr(10) || 'Question: ' || q.question AS prompt
FROM eval.contexts c, eval.questions q, range({{RUNS}}) r(run);

-- one row per attempt; rows of other models stay, so runs can be compared
CREATE TABLE IF NOT EXISTS eval.attempts (
    model VARCHAR, task_id INTEGER, attempt INTEGER,
    answer VARCHAR, query VARCHAR, result VARCHAR, error VARCHAR);

-- SQL block from an LLM answer, without trailing semicolon
CREATE OR REPLACE MACRO eval.extract_sql(answer) AS
    regexp_replace(
        CASE WHEN answer LIKE '%```%'
             THEN regexp_extract(answer, '(?i)```(?:sql|duckdb)?[ \t]*\n([\s\S]*?)```', 1)
             ELSE answer END,
        '^\s+|[\s;]+$', '', 'g');

-- Conditions B and C: table names outside of strings, quoted names and comments are not allowed
CREATE OR REPLACE MACRO eval.uses_tables(query) AS
    regexp_matches(
        regexp_replace(query, '--[^\n]*|/\*[\s\S]*?\*/|''(?:[^'']|'''')*''|"[^"]*"', '', 'g'),
        '(?i)\b(runs|run_types|duckdb_\w+|information_schema|read_\w+)\b');

CALL ossie_load('running.semantic_model.yaml');

SELECT condition, count(*) AS tasks FROM eval.tasks GROUP BY ALL ORDER BY ALL;

condition,tasks
A,40
B,40
C,40
D,40


The only Python loop: ask the LLM, run its SQL, give it one more try after an error, store every attempt. A new run replaces earlier rows of the same model.

In [46]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import monotonic, sleep

def ask_with_retry(prompt, tries=6):
    """Rate limits and timeouts of the LLM service are retried, they are no answer of the model."""
    for i in range(tries):
        try:
            return ask_llm(prompt)
        except Exception as e:
            if i == tries - 1:
                raise
            print(f"LLM call failed ({str(e)[:80]}), retry in {5 * 2 ** i} s", flush=True)
            sleep(5 * 2 ** i)

def run_task(task):
    task_id, condition, prompt = task
    cur = con.cursor()
    for attempt in (1, 2):
        answer = ask_with_retry(prompt)
        query = cur.execute("SELECT eval.extract_sql(?)", [answer]).fetchone()[0]
        try:
            if condition in "BC" and cur.execute("SELECT eval.uses_tables(?)", [query]).fetchone()[0]:
                raise ValueError("Direct table access is not allowed, use ossie_query only.")
            result, error = cur.execute(f"SELECT to_json(list(t))::VARCHAR FROM ({query}) t").fetchone()[0], None
        except Exception as e:
            result, error = None, str(e)
        cur.execute("INSERT INTO eval.attempts VALUES (?, ?, ?, ?, ?, ?, ?)",
                    [MODEL_LABEL, task_id, attempt, answer, query, result, error])
        if error is None:
            break
        prompt += f"\n\nYour previous answer:\n{answer}\n\nExecution failed:\n{error}\nFix your answer in the same format."

# A task is finished once it has a successful attempt or its second attempt. Without RESUME, the model starts over.
con.execute(f"""DELETE FROM eval.attempts WHERE model = $1 AND ({not RESUME} OR task_id NOT IN (
                  SELECT task_id FROM eval.attempts WHERE model = $1 AND (error IS NULL OR attempt = 2)))""", [MODEL_LABEL])
tasks = con.execute("""SELECT task_id, condition, prompt FROM eval.tasks
                       WHERE task_id NOT IN (SELECT task_id FROM eval.attempts WHERE model = $1)
                       ORDER BY task_id""", [MODEL_LABEL]).fetchall()
print(f"{len(tasks)} tasks to run")
started, failed = monotonic(), 0
with ThreadPoolExecutor(PARALLEL) as pool:
    futures = {pool.submit(run_task, task): task[0] for task in tasks}
    for done, future in enumerate(as_completed(futures), start=1):
        try:
            future.result()
        except Exception as e:   # task skipped, it is missing from the results
            failed += 1
            print(f"Task {futures[future]} failed: {str(e)[:120]}", flush=True)
        if done <= 3 or done % 10 == 0 or done == len(tasks):
            elapsed = monotonic() - started
            print(f"{done}/{len(tasks)} done | {elapsed / 60:.1f} min | "
                  f"about {elapsed / done * (len(tasks) - done) / 60:.1f} min left", flush=True)
print(f"{len(tasks) - failed} of {len(tasks)} tasks done with {MODEL_LABEL}")

160 tasks to run
1/160 done | 0.2 min | about 28.7 min left
2/160 done | 0.3 min | about 20.5 min left
3/160 done | 0.3 min | about 14.8 min left
10/160 done | 0.8 min | about 12.5 min left
20/160 done | 1.4 min | about 9.8 min left
30/160 done | 2.0 min | about 8.6 min left
40/160 done | 2.8 min | about 8.5 min left
50/160 done | 3.4 min | about 7.4 min left
60/160 done | 3.9 min | about 6.4 min left
70/160 done | 4.9 min | about 6.2 min left
80/160 done | 5.5 min | about 5.5 min left
90/160 done | 6.0 min | about 4.6 min left
100/160 done | 6.6 min | about 4.0 min left
110/160 done | 7.5 min | about 3.4 min left
120/160 done | 8.1 min | about 2.7 min left
130/160 done | 8.9 min | about 2.0 min left
140/160 done | 9.6 min | about 1.4 min left
150/160 done | 10.3 min | about 0.7 min left
160/160 done | 10.8 min | about 0.0 min left
160 of 160 tasks done with z-ai/glm-5.1


Scoring in SQL. Every result row becomes a list of numbers: times like `1:49:36` count in seconds and in minutes, a date also contributes its year. Then each question gets its check.

In [47]:
%%sql
CREATE OR REPLACE MACRO eval.time_sec(t) AS
    CASE WHEN len(string_split(t, ':')) = 3
         THEN string_split(t, ':')[1]::DOUBLE * 3600 + string_split(t, ':')[2]::DOUBLE * 60 + string_split(t, ':')[3]::DOUBLE
         ELSE string_split(t, ':')[1]::DOUBLE * 60 + string_split(t, ':')[2]::DOUBLE END;

-- all numbers in one JSON text
CREATE OR REPLACE MACRO eval.numbers(js) AS (
    [x::DOUBLE for x in regexp_extract_all(
        regexp_replace(regexp_replace(js, '\d{4}-\d{2}-\d{2}', '', 'g'), '\d+:\d{2}(?::\d{2}(?:\.\d+)?)?', '', 'g'),
        '-?\d+(?:\.\d+)?')]
    || [y::DOUBLE for y in regexp_extract_all(js, '(\d{4})-\d{2}-\d{2}', 1)]
    || flatten([[eval.time_sec(t), eval.time_sec(t) / 60]
                for t in regexp_extract_all(regexp_replace(js, '\d{4}-\d{2}-\d{2}', '', 'g'), '\d+:\d{2}(?::\d{2}(?:\.\d+)?)?')])
);

-- one list of numbers per result row
CREATE OR REPLACE MACRO eval.result_rows(result) AS
    [eval.numbers(r::VARCHAR) for r in json_extract(result, '$[*]')];

CREATE OR REPLACE MACRO eval.near(v, e, tol := 0.005) AS abs(v - e) <= tol * greatest(abs(e), 1e-9);

-- value:      exactly one row that contains one of the expected values
-- year_pairs: one row per expected year, and for each year a row with the year and its time
CREATE OR REPLACE MACRO eval.check(check_type, expected, tol, rows_) AS coalesce(
    CASE check_type
        WHEN 'value' THEN len(rows_) = 1
             AND list_bool_or([list_bool_or([eval.near(v, e, tol := tol) for v in rows_[1]]) for e in expected[1]])
        WHEN 'year_pairs' THEN len(rows_) = len(expected)
             AND list_bool_and([list_bool_or([list_contains(r, p[1])
                                              AND list_bool_or([eval.near(v, p[2], tol := tol) OR eval.near(v, p[3], tol := tol) for v in r])
                                              for r in rows_])
                                for p in expected])
    END, false);

CREATE OR REPLACE VIEW eval.verdicts AS
WITH last_attempt AS (
    SELECT * FROM eval.attempts
    QUALIFY row_number() OVER (PARTITION BY model, task_id ORDER BY attempt DESC) = 1
)
SELECT a.model, t.condition, t.question_id, t.run, a.attempt AS attempts, a.query, a.result, a.error,
       CASE WHEN a.error LIKE 'Invalid Input Error: ossie:%' THEN 'refused'
            WHEN a.error IS NOT NULL THEN 'error'
            WHEN a.result IS NOT NULL AND eval.check(q.check_type, q.expected, q.tol, eval.result_rows(a.result))
            THEN 'correct' ELSE 'wrong' END AS verdict   -- an empty result counts as wrong
FROM last_attempt a
JOIN eval.tasks t USING (task_id)
JOIN eval.questions q USING (question_id);

SELECT model, count(*) AS scored_answers FROM eval.verdicts GROUP BY ALL;

model,scored_answers
z-ai/glm-5.1,160
meta-llama/llama-3.1-8b-instruct,160


**Note:** the tables below show the runs made in this notebook. A run with another model or fewer repetitions will differ, and that difference is part of what the test shows.

Correct answers per model and condition after at most one retry, and how many of them were right on the first try:

In [48]:
%%sql
SELECT model, condition,
       count(*) FILTER (verdict = 'correct') || '/' || count(*) AS correct,
       count(*) FILTER (verdict = 'correct' AND attempts = 1)   AS first_try,
       count(*) FILTER (verdict = 'wrong')   AS wrong,
       count(*) FILTER (verdict = 'refused') AS refused,
       count(*) FILTER (verdict = 'error')   AS errors
FROM eval.verdicts
GROUP BY ALL
ORDER BY model, condition;

model,condition,correct,first_try,wrong,refused,errors
meta-llama/llama-3.1-8b-instruct,A,4/40,3,35,0,1
meta-llama/llama-3.1-8b-instruct,B,3/40,1,3,16,18
meta-llama/llama-3.1-8b-instruct,C,2/40,1,6,15,17
meta-llama/llama-3.1-8b-instruct,D,24/40,21,16,0,0
z-ai/glm-5.1,A,17/40,17,23,0,0
z-ai/glm-5.1,B,40/40,39,0,0,0
z-ai/glm-5.1,C,40/40,39,0,0,0
z-ai/glm-5.1,D,37/40,37,3,0,0


Correct answers per question:

In [49]:
%%sql
PIVOT (SELECT model, question_id, condition,
             count(*) FILTER (verdict = 'correct') || '/' || count(*) AS correct
      FROM eval.verdicts GROUP BY ALL)
ON condition USING first(correct)
GROUP BY model, question_id
ORDER BY model, question_id;

model,question_id,A,B,C,D
meta-llama/llama-3.1-8b-instruct,hm,0/5,0/5,0/5,4/5
meta-llama/llama-3.1-8b-instruct,hm_short,0/5,0/5,0/5,5/5
meta-llama/llama-3.1-8b-instruct,hm_times,0/5,0/5,0/5,5/5
meta-llama/llama-3.1-8b-instruct,km_2024,4/5,1/5,1/5,5/5
meta-llama/llama-3.1-8b-instruct,long_runs,0/5,0/5,0/5,4/5
meta-llama/llama-3.1-8b-instruct,pace,0/5,2/5,1/5,0/5
meta-llama/llama-3.1-8b-instruct,pace_speed,0/5,0/5,0/5,0/5
meta-llama/llama-3.1-8b-instruct,races,0/5,0/5,0/5,1/5
z-ai/glm-5.1,hm,0/5,5/5,5/5,5/5
z-ai/glm-5.1,hm_short,0/5,5/5,5/5,5/5


Everything that was not correct in the end, with the generated SQL:

In [50]:
%%sql
SELECT model, condition, question_id, verdict, query, left(error, 200) AS error
FROM eval.verdicts
WHERE verdict <> 'correct'
ORDER BY model, question_id, condition;

model,condition,question_id,verdict,query,error
meta-llama/llama-3.1-8b-instruct,A,hm,wrong,SELECT COUNT(*) FROM runs AS r JOIN run_types AS rt ON r.run_type_id = rt.run_type_id WHERE rt.run_type = 'half marathon' AND r.run_type_id IN (SELECT run_type_id FROM run_types WHERE category = 'marathon' AND run_type = 'half marathon'),None
meta-llama/llama-3.1-8b-instruct,A,hm,wrong,SELECT COUNT(*) FROM runs WHERE run_type_id = (SELECT run_type_id FROM run_types WHERE run_type = 'Half Marathon'),None
meta-llama/llama-3.1-8b-instruct,A,hm,wrong,SELECT COUNT(*) FROM runs WHERE run_type_id IN (SELECT run_type_id FROM run_types WHERE run_type = 'Half Marathon'),None
meta-llama/llama-3.1-8b-instruct,A,hm,wrong,SELECT COUNT(*) AS num_half_marathons FROM runs WHERE run_type_id IN (SELECT run_type_id FROM run_types WHERE run_type = 'half marathon'),None
meta-llama/llama-3.1-8b-instruct,A,hm,wrong,SELECT COUNT(run_id) as num_half_marathonsFROM runsJOIN run_types ON runs.run_type_id = run_types.run_type_idWHERE run_types.run_type = 'Half Marathon',None
meta-llama/llama-3.1-8b-instruct,B,hm,refused,"SELECT run_type, COUNT(*) as run_count FROM ossie_query(['run_count'], ['run_types.run_type'], ['run_types.run_type = ""Half marathon""']) GROUP BY run_type","Invalid Input Error: ossie: filter ""run_types.run_type = ""Half marathon"""" references ""Half marathon"", which is neither a metric nor a dataset.field name.Did you mean: ""run_count""LINE 1: ...))::VARC"
meta-llama/llama-3.1-8b-instruct,B,hm,error,"SELECT run_type AS run_type, ossie_query(['run_count'], ['year', 'run_type'], ['category = ""Training""]) AS training, ossie_query(['run_count'], ['year', 'run_type'], ['category = ""Race""]) AS raceFROM ossie_query(['run_type', 'category'], [], ['run_types.run_type = ""Half marathon"" AND run_types.category IN (""Training"", ""Race"")'])","Parser Error: syntax error at or near ""run_count""LINE 1: ...'], ['category = ""Training""]) AS training, ossie_query(['run_count'], ['year', 'run_type'], ['category = ""Race""]..."
meta-llama/llama-3.1-8b-instruct,B,hm,refused,"SELECT COALESCE(SUM(o113.run_type = 'Half marathon'), 0) AS run_countFROM ossie_query(['run_types.run_type'], ['run_types.run_type'], ['run_types.run_type = ""Half marathon""']) o113","Invalid Input Error: ossie: model ""running"" has no metric named ""run_types.run_type"".Did you mean: ""run_count""LINE 2: FROM ossie_query(['run_types.run_type'], ['run_types.run_type... ^"
meta-llama/llama-3.1-8b-instruct,B,hm,refused,"SELECT COUNT(*) AS run_countFROM ossie_query( ['COUNT(runs.run_type_id)'], ['runs.run_type'], ['runs.run_type = ""Half marathon""'])","Invalid Input Error: ossie: model ""running"" has no metric named ""COUNT(runs.run_type_id)"".Did you mean: ""run_count""LINE 2: FROM ossie_query( ^^^^^^^^^^^^"
meta-llama/llama-3.1-8b-instruct,B,hm,error,"SELECT run_type_id, run_type, COUNT(*) FROM ossie_query(['run_count'], ['runtypes.run_type_id'], ['runtypes.run_type = ""Half marathon""']) AS rJOIN run_types AS rt ON r.run_type_id = rt.run_type_id","Direct table access is not allowed, use ossie_query only."


Every failed attempt, including those the second try repaired:

In [51]:
%%sql
SELECT a.model, t.condition, t.question_id, a.attempt, a.query, left(a.error, 200) AS error
FROM eval.attempts a JOIN eval.tasks t USING (task_id)
WHERE a.error IS NOT NULL
ORDER BY a.model, t.question_id, t.condition;

model,condition,question_id,attempt,query,error
meta-llama/llama-3.1-8b-instruct,B,hm,1,"SELECT COUNT(runs.run_id) AS half_marathonsFROM ossie_query(['run_count'], [], ['run_types.run_type = ""Half marathon'])","Direct table access is not allowed, use ossie_query only."
meta-llama/llama-3.1-8b-instruct,B,hm,2,"SELECT run_type AS run_type, ossie_query(['run_count'], ['year', 'run_type'], ['category = ""Training""]) AS training, ossie_query(['run_count'], ['year', 'run_type'], ['category = ""Race""]) AS raceFROM ossie_query(['run_type', 'category'], [], ['run_types.run_type = ""Half marathon"" AND run_types.category IN (""Training"", ""Race"")'])","Parser Error: syntax error at or near ""run_count""LINE 1: ...'], ['category = ""Training""]) AS training, ossie_query(['run_count'], ['year', 'run_type'], ['category = ""Race""]..."
meta-llama/llama-3.1-8b-instruct,B,hm,2,"SELECT run_type, COUNT(*) as run_count FROM ossie_query(['run_count'], ['run_types.run_type'], ['run_types.run_type = ""Half marathon""']) GROUP BY run_type","Invalid Input Error: ossie: filter ""run_types.run_type = ""Half marathon"""" references ""Half marathon"", which is neither a metric nor a dataset.field name.Did you mean: ""run_count""LINE 1: ...))::VARC"
meta-llama/llama-3.1-8b-instruct,B,hm,2,"SELECT run_type_id, run_type, COUNT(*) FROM ossie_query(['run_count'], ['runtypes.run_type_id'], ['runtypes.run_type = ""Half marathon""']) AS rJOIN run_types AS rt ON r.run_type_id = rt.run_type_id","Direct table access is not allowed, use ossie_query only."
meta-llama/llama-3.1-8b-instruct,B,hm,1,"SELECT SUM(runs.run_types.run_type_id) AS run_countFROM ossie_query( ['SUM(runs.run_type_id)'], ['runs.year'], ['runs.run_type = ""Half marathon""'])","Direct table access is not allowed, use ossie_query only."
meta-llama/llama-3.1-8b-instruct,B,hm,1,"SELECT r.run_type_id, rt.run_type, COUNT(*) FROM ossie_query(['run_count'], ['runs.run_type_id AS run_type_id', 'r.run_type_id'], ['r.run_type_id = rt.run_type_id', 'rt.run_type = ""Half marathon""']) AS rJOIN ossie_query(['runs.run_type_id'], ['run_types.run_type_id AS run_type_id'], []) AS rt ON r.run_type_id = rt.run_type_idGROUP BY r.run_type_id, rt.run_type","Invalid Input Error: ossie: dimension ""r.run_type_id"" names dataset ""r"", which the model does not declareLINE 2: FROM ossie_query(['run_count'], ['runs.run_type_id AS run_type_id... ^^^"
meta-llama/llama-3.1-8b-instruct,B,hm,2,"SELECT COALESCE(SUM(o113.run_type = 'Half marathon'), 0) AS run_countFROM ossie_query(['run_types.run_type'], ['run_types.run_type'], ['run_types.run_type = ""Half marathon""']) o113","Invalid Input Error: ossie: model ""running"" has no metric named ""run_types.run_type"".Did you mean: ""run_count""LINE 2: FROM ossie_query(['run_types.run_type'], ['run_types.run_type... ^"
meta-llama/llama-3.1-8b-instruct,B,hm,2,"SELECT COUNT(*) AS run_countFROM ossie_query( ['COUNT(runs.run_type_id)'], ['runs.run_type'], ['runs.run_type = ""Half marathon""'])","Invalid Input Error: ossie: model ""running"" has no metric named ""COUNT(runs.run_type_id)"".Did you mean: ""run_count""LINE 2: FROM ossie_query( ^^^^^^^^^^^^"
meta-llama/llama-3.1-8b-instruct,B,hm,1,"SELECT run_types.run_type, COUNT(runs.run_id) as run_count FROM ossie_query(['run_count'], ['run_types.run_type'], ['runs.run_type_id = run_types.run_type_id', 'run_types.run_type = ""Half marathon""']) GROUP BY run_types.run_type","Direct table access is not allowed, use ossie_query only."
meta-llama/llama-3.1-8b-instruct,B,hm,1,"SELECT COALESCE(SUM(o113.run_type = 'Half marathon'), 0) AS run_countFROM ossie_query(['run_count'], ['run_types.run_type'], ['run_types.run_type = ""Half marathon""']) o113","Invalid Input Error: ossie: filter ""run_types.run_type = ""Half marathon"""" references ""Half marathon"", which is neither a metric nor a dataset.field name.Did you mean: ""run_count""LINE 2: FROM ossie_"


All attempts as CSV, for example to compare models outside the notebook:

In [52]:
%%sql
COPY (SELECT a.model, t.condition, t.question_id, t.run, a.attempt, a.query, a.result, a.error
      FROM eval.attempts a JOIN eval.tasks t USING (task_id)
      ORDER BY ALL)
TO 'llm_attempts.csv' (HEADER);

Count
405
